# Attention interpretability walk-through

This notebook loads the best trained checkpoint and renders three families of attention maps for a chosen sentence:

* **Encoder self-attention** — how the model relates English tokens to each other.
* **Decoder self-attention** — strictly causal, shows how each generated Hindi token attends to previously generated ones.
* **Encoder → decoder cross-attention** — the bridge between the two languages and the most interpretable family. Strong diagonals here mean the model has learned a soft alignment.

Each figure is a grid of `layers × heads`. Different heads learn surprisingly different patterns — some learn positional alignment, some focus on punctuation, and some on specific lexical anchors (numbers, named entities).

In [ ]:
import numpy as np
import torch
from tokenizers import Tokenizer

from config import get_best_weights_path, get_config
from decode import greedy_decode
from model import build_transformer
from attention_heatmap import plot_attention_grid, plot_attention_single

In [ ]:
cfg = get_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok_src = Tokenizer.from_file(cfg['tokenizer_file'].format(cfg['lang_src']))
tok_tgt = Tokenizer.from_file(cfg['tokenizer_file'].format(cfg['lang_tgt']))

model = build_transformer(
    tok_src.get_vocab_size(), tok_tgt.get_vocab_size(),
    cfg['max_seq_len'], cfg['max_seq_len'],
    d_model=cfg['d_model'], N=cfg['n_layers'], h=cfg['n_heads'],
    dropout=cfg['dropout'], d_ff=cfg['d_ff'],
).to(device)
state = torch.load(get_best_weights_path(cfg), map_location=device, weights_only=False)
model.load_state_dict(state['model_state_dict'])
model.eval()
print(f"Loaded checkpoint (step={state.get('global_step', '?')}, chrF++={state.get('best_chrf', '?')})")

In [ ]:
SENTENCE = 'The manufacture or processing of goods.'

sos = tok_src.token_to_id('[SOS]')
eos = tok_src.token_to_id('[EOS]')
pad = tok_src.token_to_id('[PAD]')
ids = tok_src.encode(SENTENCE).ids[: cfg['max_seq_len'] - 2]
tokens = [sos] + ids + [eos] + [pad] * (cfg['max_seq_len'] - len(ids) - 2)
src = torch.tensor(tokens, dtype=torch.int64, device=device).unsqueeze(0)
src_mask = (src != pad).unsqueeze(0).unsqueeze(0).int()

with torch.no_grad():
    out_ids = greedy_decode(model, src, src_mask, tok_src, tok_tgt, cfg['max_seq_len'], device).tolist()

src_tokens = [tok_src.id_to_token(i) for i in [sos] + ids + [eos]]
tgt_tokens = [tok_tgt.id_to_token(i) for i in out_ids if i not in (tok_tgt.token_to_id('[PAD]'),)]
print('Source :', SENTENCE)
print('Output :', tok_tgt.decode([i for i in out_ids if i not in (tok_tgt.token_to_id('[SOS]'), tok_tgt.token_to_id('[EOS]'))]))

In [ ]:
def stack(kind):
    """Return {layer: (heads, T_out, T_in) ndarray} for the chosen attention family."""
    out = {}
    for li, layer in enumerate(model.decoder.layers if kind != 'encoder' else model.encoder.layers):
        if kind == 'encoder':
            attn = layer.self_attention_block.attention_scores
        elif kind == 'decoder_self':
            attn = layer.self_attention_block.attention_scores
        else:
            attn = layer.cross_attention_block.attention_scores
        out[li] = attn[0].detach().float().cpu().numpy()
    return out

# Cross-attention: how each Hindi token aligns to the English source
cross = stack('cross')
plot_attention_grid({0: cross[0], cfg['n_layers'] - 1: cross[cfg['n_layers'] - 1]},
                    src_tokens=src_tokens, tgt_tokens=tgt_tokens[:len(src_tokens)+4],
                    title=f'Cross-attention: "{SENTENCE}"',
                    out_path='results/visualizations/encoder-decoder.png')

In [ ]:
# Decoder self-attention — strictly lower-triangular (causal)
dec_self = stack('decoder_self')
plot_attention_grid({0: dec_self[0], cfg['n_layers'] - 1: dec_self[cfg['n_layers'] - 1]},
                    src_tokens=tgt_tokens[:20], tgt_tokens=tgt_tokens[:20],
                    title='Decoder self-attention',
                    out_path='results/visualizations/decoder.png')

## How to read these maps

* The **y axis** is the prediction (each row is one generated Hindi token).
* The **x axis** is the source (English tokens) for cross-attention, or the prefix already generated for decoder self-attention.
* **Brighter = higher attention weight.** A row that's almost a delta at a single column means "to generate this Hindi token the head looked almost exclusively at that one English token."

Patterns to look for:
* **Diagonal cross-attention** in mid layers — soft alignment between source and target order.
* **Sharp columns on `[EOS]`** in early decoding steps — common warm-up pattern.
* **Heads that activate on rare lexical items** (numbers, named entities) — the model leaning on copy-style behavior.
* **Triangular structure in decoder self-attention** — confirmation that the causal mask is in effect.